# 03. Интерактивный отчёт и дашборд: plotly

In [ ]:
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

from style import (
     BLUE, CATEGORY_PALETTE, CLUSTER_PALETTE, ERROR_RED, GRAY, GRAY_LIGHT,
     GREEN, GREEN_DARK, GREEN_LIGHT, MINT
)

pio.templates.default = "plotly_white"
pio.templates["plotly_white"].layout.colorway = CATEGORY_PALETTE
px.defaults.color_discrete_sequence = CATEGORY_PALETTE

REPORT_DIR = Path("reports")
REPORT_DIR.mkdir(exist_ok=True)

ASSETS_DIR = REPORT_DIR / Path('plotly_assets')
ASSETS_DIR.mkdir(exist_ok=True)

df = pd.read_pickle("data/sales_clean.pkl")

orders = (df.groupby("ORDERNUMBER")
            .agg(ORDERDATE=("ORDERDATE", "first"),
                 CUSTOMERNAME=("CUSTOMERNAME", "first"),
                 COUNTRY=("COUNTRY", "first"),
                 STATUS=("STATUS", "first"),
                 STATUS_GROUP=("STATUS_GROUP", "first"),
                 SALES_EST=("SALES_EST", "sum"),
                 LINES=("SALES_EST", "size"),
                 MAX_QTY=("QUANTITYORDERED", "max"))
            .reset_index())
orders["STATUS"] = orders["STATUS"].astype(str)

PERIOD = f"{df['ORDERDATE'].min():%d.%m.%Y}–{df['ORDERDATE'].max():%d.%m.%Y}"
SOURCE = f"check.csv, {PERIOD}, {len(orders)} заказов · выручка — нижняя оценка"
print(SOURCE)

## Функции оформления

In [ ]:
def style_layout(fig, title, subtitle=SOURCE, height=480):
    """Заголовок-вывод, подзаголовок с источником, единые размеры и формат чисел при наведении."""
    fig.update_layout(
        title={"text": f"<b>{title}</b><br><sup>{subtitle}</sup>", "x": 0.01},
        height=height,
        margin={"l": 20, "r": 20, "t": 90, "b": 40},
        hoverlabel={"namelength": -1},
        separators=". ",           # десятичная точка, пробел — разделитель тысяч
        legend_title_text="",
    )
    return fig


STATUS_ORDER = ["Shipped", "Resolved", "In Process", "Disputed", "On Hold", "Cancelled"]
PURPLE = CATEGORY_PALETTE[5]   # фиолетовый из палитры style.py
ORANGE = CLUSTER_PALETTE[5]    # оранжевый из палитры style.py
STATUS_COLORS = dict(zip(STATUS_ORDER, [GRAY_LIGHT, GRAY, BLUE, PURPLE, ORANGE, ERROR_RED]))

## 1. Временной ряд, который можно разглядывать

`rangeslider` - при наведении видно не только выручку, но и число заказов и средний заказ за месяц: **три метрики на одном графике без второй оси Y**.

In [ ]:
monthly = orders.assign(
    YEAR_MONTH=orders["ORDERDATE"].dt.to_period("M").dt.to_timestamp()
).groupby("YEAR_MONTH").agg(
    SALES_EST=("SALES_EST", "sum"), 
    orders=("SALES_EST", "size"),
    avg_order=("SALES_EST", "mean")
).reset_index()

fig = px.line(
    monthly, x="YEAR_MONTH", y="SALES_EST", markers=True,
    hover_data={"SALES_EST": ":,.0f", "orders": True, "avg_order": ":,.0f"},
    labels={
        "YEAR_MONTH": "Месяц", "SALES_EST": "Выручка", 
        "orders": "Заказов", "avg_order": "Средний заказ"
    },
)
fig.update_xaxes(
    rangeslider_visible=False,
    rangeselector={"buttons": [
        {"count": 6, "label": "6 мес", "step": "month", "stepmode": "backward"},
        {"count": 1, "label": "1 год", "step": "year", "stepmode": "backward"},
        {"step": "all", "label": "всё"},
    ]},
)
fig.update_yaxes(tickformat="~s", rangemode="tozero")
style_layout(fig, "Ноябрьские пики повторяются каждый год")
fig.write_html(ASSETS_DIR / 'sales_time_series.html')
# fig.show()

выделите мышью ноябрь 2004 и наведите на точку — выручка растёт за счёт **числа заказов**, а средний заказ почти такой же, как в другие месяцы. Это подтверждает гипотезу H2 из `02`: бизнес растёт частотой заказов, а не их размером.

## 2. Фильтр без Dash: выпадающий список

Кнопки и выпадающие списки (`updatemenus`) переключают видимость линий прямо в HTML — без сервера.
Функция универсальна: разрез передаём параметром.

In [ ]:
def line_with_dropdown(data, by, value="SALES_EST", title=""):
    """Помесячная динамика по категориям столбца `by` + выпадающий список для выбора категории."""
    wide = data.pivot_table(
        index="YEAR_MONTH", columns=by, values=value,
        aggfunc="sum", fill_value=0, observed=True
    )
    fig = go.Figure()
    for name in wide.columns:
        fig.add_trace(go.Scatter(
            x=wide.index, y=wide[name], name=str(name), mode="lines+markers",
            hovertemplate="%{x|%b %Y}: %{y:,.0f}<extra>" + str(name) + "</extra>"
        ))

    n = len(wide.columns)
    buttons = [{"label": "Все", "method": "update", "args": [{"visible": [True] * n}]}]
    for i, name in enumerate(wide.columns):
        buttons.append({
            "label": str(name), "method": "update",
            "args": [{"visible": [j == i for j in range(n)]}]
        })

    fig.update_layout(updatemenus=[{
        "buttons": buttons, "direction": "down",
        "x": 1, "xanchor": "right", "y": 1.12, "yanchor": "bottom"
    }])
    fig.update_yaxes(tickformat="~s", rangemode="tozero", title="Выручка")
    return style_layout(fig, title)


fig = line_with_dropdown(
    df, by="PRODUCTLINE",
    title="Сезонный ноябрьский пик есть у каждой продуктовой линии"
)
fig.write_html(ASSETS_DIR / 'productline_sales_series.html')
# fig.show()

In [ ]:
fig = line_with_dropdown(
    df, by="TERRITORY", 
    title="Динамика выручки по территориям продаж"
)
fig.write_html(ASSETS_DIR / 'territory_sales_series.html')
# fig.show()

**Попробуйте:** выберите в списке по одной линии. Ноябрь — максимум у всех семи линий, значит, сезонность — свойство **спроса клиентов** (подарочный сезон), а не одного товара.

## 3. клиенты

Главное преимущество интерактива для аналитика — **hover**: не нужно подписывать 92 точки, достаточно навести мышь.
Шкала Y логарифмическая — выручка клиентов различается в десятки раз, и это указано в подписи оси.

In [ ]:
customers = df.groupby("CUSTOMERNAME").agg(
     SALES_EST=("SALES_EST", "sum"),
     n_orders=("ORDERNUMBER", "nunique"),
     COUNTRY=("COUNTRY", "first"),
     TERRITORY=("TERRITORY", "first")
).reset_index()
customers["avg_order"] = customers["SALES_EST"] / customers["n_orders"]
customers["TERRITORY"] = customers["TERRITORY"].astype(str)

fig = px.scatter(
     customers, x="n_orders", y="SALES_EST", size="avg_order", color="TERRITORY",
     hover_name="CUSTOMERNAME",
     hover_data={"COUNTRY": True, "SALES_EST": ":,.0f", "avg_order": ":,.0f", "TERRITORY": False},
     log_y=True, size_max=22,
     labels={
          "n_orders": "Число заказов за период", "SALES_EST": "Выручка (лог. шкала)",
          "avg_order": "Средний заказ", "COUNTRY": "Страна"
     },
)
fig.update_yaxes(tickformat="~s")
style_layout(
    fig, 
    "Два клиента далеко впереди: у них в 6–9 раз больше заказов, чем у типичного клиента"
)
fig.write_html(ASSETS_DIR / 'n_orders_sales_scatter.html')
# fig.show()

Два клиента справа вверху — **Euro Shopping Channel** и **Mini Gifts Distributors** (наведите на точки). Остальные клиенты сделали 1–5 заказов.
Размер точки (средний заказ) у них обычный: они крупные не из-за «больших заказов», а из-за **частоты**.

> **Про данные внутри HTML.** Фигура plotly хранит все значения внутри себя (см. размер JSON выше). Поэтому перед построением **агрегируем** данные до нужного уровня: так файл меньше.

## 4. География и структура

Карта и treemap — классические интерактивные графики.
Но помним про точность восприятия: **цвет и площадь** отвечают на вопрос «больше или меньше», а не «во сколько раз». Точные сравнения — в столбиках (`02`).

In [ ]:
COUNTRY_ISO3 = {
    "USA": "USA", "Spain": "ESP", "France": "FRA", "Australia": "AUS", "UK": "GBR", "Italy": "ITA",
    "Finland": "FIN", "Norway": "NOR", "Singapore": "SGP", "Canada": "CAN", "Denmark": "DNK",
    "Germany": "DEU", "Sweden": "SWE", "Austria": "AUT", "Japan": "JPN", "Belgium": "BEL",
    "Switzerland": "CHE", "Philippines": "PHL", "Ireland": "IRL",
}

by_country = orders.groupby("COUNTRY", observed=True).agg(
    SALES_EST=("SALES_EST", "sum"), 
    n_orders=("SALES_EST", "size")
).reset_index()
by_country["COUNTRY"] = by_country["COUNTRY"].astype(str)
by_country["ISO3"] = by_country["COUNTRY"].map(COUNTRY_ISO3)
assert by_country["ISO3"].notna().all(), "есть страна без ISO-кода"
by_country["share"] = by_country["SALES_EST"] / by_country["SALES_EST"].sum()

fig = px.choropleth(
    by_country, locations="ISO3", color="SALES_EST", hover_name="COUNTRY",
    hover_data={"ISO3": False, "SALES_EST": ":,.0f", "share": ":.1%", "n_orders": True},
    color_continuous_scale=[MINT, GREEN, GREEN_DARK], projection="natural earth",
    labels={"SALES_EST": "Выручка", "share": "Доля", "n_orders": "Заказов"},
)
top_country = by_country.nlargest(1, "SALES_EST").iloc[0]
style_layout(
    fig, 
    f"{top_country['COUNTRY']} — {top_country['share']:.0%} выручки; далее Испания и Франция",
    height=500
)
fig.write_html(ASSETS_DIR / 'country_stats.html')
# fig.show()

> Подложка карты загружается браузером из интернета. В закрытом контуре карта может не отрисоваться — тогда используйте столбики.

In [ ]:
tree = df.groupby(
    ["TERRITORY", "COUNTRY", "PRODUCTLINE"], observed=True
)["SALES_EST"].sum().reset_index()
tree[["TERRITORY", "COUNTRY", "PRODUCTLINE"]] = tree[["TERRITORY", "COUNTRY", "PRODUCTLINE"]].astype(str)

fig = px.treemap(
    tree, path=[px.Constant("Все продажи"), "TERRITORY", "COUNTRY", "PRODUCTLINE"],
    values="SALES_EST", color="TERRITORY"
)
fig.update_traces(
    texttemplate="%{label}<br>%{percentRoot:.1%}",
    hovertemplate="%{id}<br>Выручка: %{value:,.0f}<br>Доля: %{percentRoot:.1%}<extra></extra>"
)
style_layout(
    fig, 
    "Структура продаж: территория → страна → продуктовая линия",
    height=600
)

fig.write_html(ASSETS_DIR / 'treemap_country_sales.html')
# fig.show()

Treemap полезен для **навигации по иерархии**: кликните на `EMEA` → `Spain`, чтобы увидеть структуру линий внутри страны.
Напомним про территории из `01`: `NA` — это North America, а Сингапур разнесён по двум территориям (`Japan` и `APAC`) — это настройка компании, а не ошибка.

## 5. Заказы: найти конкретные проблемные

Box plot с точками: коробка — обычный разброс, точки — **каждый заказ**. При наведении видны номер заказа, клиент и дата — то, что нужно, чтобы передать список в отдел продаж.

In [ ]:
fig_orders = px.box(
    orders, x="STATUS", y="SALES_EST",
    color="STATUS", points="all",
    category_orders={"STATUS": STATUS_ORDER}, 
    color_discrete_map=STATUS_COLORS,
    hover_data={
        "ORDERNUMBER": True, "CUSTOMERNAME": True, 
        "ORDERDATE": "|%d.%m.%Y",
        "MAX_QTY": True, "STATUS": False
    },
    labels={
        "STATUS": "Статус", "SALES_EST": "Сумма заказа", 
        "ORDERNUMBER": "Заказ",
        "CUSTOMERNAME": "Клиент", "ORDERDATE": "Дата", 
        "MAX_QTY": "Макс. кол-во в строке"
    }
)
# цвет = та же ось X, группировка не нужна
fig_orders.update_layout(showlegend=False, boxmode="overlay")
fig_orders.update_yaxes(tickformat="~s", rangemode="tozero")
style_layout(
    fig_orders, 
    "Заказы On Hold и Cancelled крупнее обычных — наведите, чтобы увидеть клиента и дату"
)
# fig_orders.show()
fig_orders.write_html(ASSETS_DIR / 'box_sales_orders.html')

orders.query("STATUS in ['On Hold', 'Cancelled']").sort_values("ORDERDATE")[
    ["ORDERNUMBER", "ORDERDATE", "CUSTOMERNAME", "STATUS", "SALES_EST", "MAX_QTY"]
]

Интерактив позволил сразу перейти от гипотезы H4 («крупные заказы застревают») к **конкретному списку**: 4 заказа `On Hold` в апреле–мае 2005 (среди них — заказы Tekni Collectables и The Sharp Gifts Warehouse с количеством 76–85 шт. в строке) и 4 отмены в 2003–2004.

---
## 6. Интерактивный дашборд

Собираем главное на одной странице через `make_subplots`: сверху — KPI-индикаторы, ниже — четыре панели.
Индикатор `number+delta` сам показывает рост к прошлому году. Дашборд строится функцией — его можно пересобрать для любого среза данных.

Долю по размеру сделки показываем кольцевой диаграммой: это допустимо, потому что долей всего **3, они сильно различаются, а в сумме дают осмысленные 100%**.

In [ ]:
def hbar(values: pd.Series, color=BLUE, highlight_color=GREEN):
    """Горизонтальный столбчатый трейс: сортировка, подписи, выделение лидера."""
    s = values.sort_values()
    colors = [highlight_color if name == s.idxmax() else color for name in s.index]
    return go.Bar(
        x=s.values, y=s.index.astype(str), orientation="h", marker_color=colors,
        text=[f"{v / 1e6:.2f} млн" if v >= 1e6 else f"{v / 1e3:.0f} тыс" for v in s.values],
        textposition="outside", cliponaxis=False,
        hovertemplate="%{y}: %{x:,.0f}<extra></extra>"
    )


def build_dashboard(data: pd.DataFrame, title="Продажи: интерактивный дашборд"):
    ord_ = data.groupby("ORDERNUMBER").agg(
        SALES_EST=("SALES_EST", "sum"),
        STATUS_GROUP=("STATUS_GROUP", "first"),
        ORDERDATE=("ORDERDATE", "first")
    )
    jan_may = ord_[ord_["ORDERDATE"].dt.month <= 5]
    ytd = jan_may.groupby(jan_may["ORDERDATE"].dt.year)["SALES_EST"].sum()

    fig = make_subplots(
        rows=3, cols=3, row_heights=[0.16, 0.42, 0.42], 
        vertical_spacing=0.09, horizontal_spacing=0.1,
        specs=[
            [{"type": "indicator"}, {"type": "indicator"}, {"type": "indicator"}],
            [{"colspan": 2}, None, {}],
            [{}, {"type": "domain"}, {}]
        ],
        subplot_titles=[
            "", "", "",
            "Выручка по месяцам", "Продуктовые линии",
            "Топ-8 стран", "Размер сделки: доля выручки", "Неисполненные заказы, шт."
        ],
    )

    # KPI
    fig.add_trace(go.Indicator(
        mode="number", value=data["SALES_EST"].sum(),
        number={"valueformat": ",.0f"}, title={"text": "Выручка за период"}
    ), row=1, col=1)
    if len(ytd) >= 2:
        fig.add_trace(go.Indicator(
            mode="number+delta", value=ytd.iloc[-1],
            number={"valueformat": ",.0f"},
            delta={
                "reference": ytd.iloc[-2], "relative": True, "valueformat": "+.0%",
                "increasing": {"color": GREEN}, "decreasing": {"color": ERROR_RED}
            },
            title={"text": f"Январь–май {ytd.index[-1]} к {ytd.index[-2]}"}
        ), row=1, col=2)
    fig.add_trace(go.Indicator(
        mode="number", value=ord_["SALES_EST"].mean(),
        number={"valueformat": ",.0f"}, title={"text": "Средний заказ"}
    ), row=1, col=3)

    # Динамика
    ts = data.groupby("YEAR_MONTH")["SALES_EST"].sum()
    fig.add_trace(go.Scatter(
        x=ts.index, y=ts.values, mode="lines+markers", fill="tozeroy",
        hovertemplate="%{x|%b %Y}: %{y:,.0f}<extra></extra>"
    ), row=2, col=1)

    # Продуктовые линии и страны
    fig.add_trace(hbar(data.groupby("PRODUCTLINE", observed=True)["SALES_EST"].sum()), row=2, col=3)
    fig.add_trace(hbar(data.groupby("COUNTRY", observed=True)["SALES_EST"].sum().nlargest(8)), row=3, col=1)

    # Размер сделки
    deal = data.groupby("DEALSIZE", observed=True)["SALES_EST"].sum()
    fig.add_trace(go.Pie(
        labels=deal.index.astype(str), values=deal.values, hole=0.55, sort=False,
        textinfo="label+percent", marker_colors=[GREEN_LIGHT, GREEN, GREEN_DARK],
        hovertemplate="%{label}: %{value:,.0f}<extra></extra>"
    ), row=3, col=2)

    # Неисполненные заказы
    status = ord_["STATUS_GROUP"].value_counts().reindex(["In progress", "Problem", "Cancelled"], fill_value=0)
    fig.add_trace(go.Bar(
        x=status.index.astype(str), y=status.values, text=status.values, textposition="outside",
        cliponaxis=False, marker_color=[BLUE, ORANGE, ERROR_RED],
        hovertemplate="%{x}: %{y} заказов<extra></extra>"
    ), row=3, col=3)

    fig.update_xaxes(tickformat="~s", row=2, col=3)
    fig.update_xaxes(tickformat="~s", row=3, col=1)
    fig.update_yaxes(tickformat="~s", rangemode="tozero", row=2, col=1)
    fig.update_layout(showlegend=False)

    period = f"{data['ORDERDATE'].min():%m.%Y}–{data['ORDERDATE'].max():%m.%Y}"
    n_done = ord_["STATUS_GROUP"].eq("Completed").sum()
    return style_layout(
        fig, f"{title} · {period}",
        subtitle=f"{SOURCE} · исполнено {n_done} из {len(ord_)} заказов", height=950
    )


dashboard = build_dashboard(df)
# dashboard.show()

## 7. Сохранение: один HTML на весь отчёт

- `include_plotlyjs="cdn"` — файл маленький, но при открытии нужен интернет;
- `include_plotlyjs=True` — файл большой, но работает офлайн (для закрытого контура).

In [ ]:
def save_report_html(figs, path, title, include_plotlyjs="cdn", verbose=True):
    """Несколько фигур plotly -> одна HTML-страница (plotly.js подключается один раз)."""
    parts = [
        fig.to_html(
            full_html=False, 
            include_plotlyjs=(include_plotlyjs if i == 0 else False)
        )
        for i, fig in enumerate(figs)
    ]
    html = (f"<html><head><meta charset='utf-8'><title>{title}</title></head>"
            f"<body style='font-family: sans-serif; max-width: 1200px; margin: auto'>"
            f"<h1>{title}</h1>{''.join(parts)}</body></html>")
    Path(path).write_text(html, encoding="utf-8")
    if verbose:
        print(f"Сохранено: {path} ({Path(path).stat().st_size / 1e3:.0f} КБ)")


dashboard.write_html(
    REPORT_DIR / "dashboard_plotly.html", 
    include_plotlyjs="cdn"
)

report_figs = [
    dashboard,
    line_with_dropdown(df, by="PRODUCTLINE", title="Динамика по продуктовым линиям"),
    fig_orders,
]
save_report_html(
    report_figs, REPORT_DIR / "sales_report_plotly.html", 
    "Продажи 2003–2005: интерактивный отчёт"
)

## Итоги

**Что дал интерактив по сравнению со статичным отчётом `02`:**

1. **Hover** превратил выводы в конкретные объекты: какие именно клиенты лидируют, какие заказы застряли в `On Hold`.
3. **Выпадающий список** подтвердил, что ноябрьский пик есть у всех продуктовых линий, — поддержка H1.
4. **Treemap и карта** — удобная навигация по структуре, но точные сравнения остаются за столбиками.

**Чего plotly не заменяет:**

- картинку в документ или на печать всё равно лучше делать, используя matplotlib;
- HTML-файл содержит данные графиков;
- в отчёте, который читают один раз сверху вниз, интерактив никто не использует — там важнее заголовок-вывод.